# Sprint 5 · Sesión práctica consolidada · Limpieza, groupby y visualización con pandas

**Duración:** 70 minutos  
**Modalidad:** Google Colab  
**Dataset usado:** `sprint05_sesion_practica_licores_latam.csv`

Esta sesión aplica Python y pandas sobre un caso de licores latinoamericanos. El objetivo no es memorizar sintaxis, sino seguir un flujo de análisis: cargar, explorar, limpiar, resumir, visualizar y comunicar hallazgos.


## Objetivos de la sesión práctica

Al finalizar, el estudiante podrá:

1. Cargar un archivo CSV en Google Colab usando `pandas`.
2. Explorar la estructura de un dataset con `.head()`, `.info()`, `.shape` y `.describe()`.
3. Detectar valores faltantes con `.isna().sum()`.
4. Aplicar estrategias básicas de limpieza: eliminar, imputar y crear copias de trabajo.
5. Detectar outliers con el método IQR.
6. Usar `groupby()` para responder preguntas analíticas.
7. Crear visualizaciones simples con `.plot()`.


## Agenda sugerida · 70 minutos

| Minutos | Bloque | Propósito |
|---:|---|---|
| 0–5 | Apertura | Recordar Colab, pandas y DataFrame |
| 5–12 | Carga de datos | Leer el CSV y validar que cargó correctamente |
| 12–20 | Exploración inicial | Revisar filas, columnas, tipos y estadísticas |
| 20–35 | Tratamiento de nulos | Contar nulos, revisar filas afectadas e imputar |
| 35–45 | Outliers con IQR | Calcular límites e identificar precios extremos |
| 45–58 | `groupby()` | Responder preguntas por país, categoría y canal |
| 58–65 | Visualización | Crear gráficas simples con pandas |
| 65–70 | Cierre | Validación, takeaways y canales de ayuda |


## 0. Recordatorio: cómo trabajar en Colab

Antes de ejecutar el notebook:

1. Abre el panel izquierdo de archivos.
2. Sube el archivo `sprint05_sesion_practica_licores_latam.csv`.
3. Ejecuta las celdas en orden.
4. Si aparece un error, lee el mensaje completo antes de modificar el código.

Un error común en Colab es ejecutar una celda que depende de una variable creada en una celda anterior. Si eso ocurre, ejecuta el notebook desde el inicio.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)


## Ejercicio 1 · Cargar y validar el dataset

Carga el archivo con `pd.read_csv()` y revisa las primeras filas.

Pregunta base: **¿qué representa cada fila del dataset?**


In [ ]:
ruta = "sprint05_sesion_practica_licores_latam.csv"

df = pd.read_csv(ruta)
df.head()


In [ ]:
# Dimensiones del dataset
print("Filas y columnas:", df.shape)

# Nombres de columnas
list(df.columns)


### Preguntas guiadas

- ¿La tabla parece estar a nivel de producto?
- ¿Qué columnas son categóricas?
- ¿Qué columnas son numéricas?
- ¿Qué columnas podrían servir para responder preguntas de negocio?


## Ejercicio 2 · Exploración inicial

Antes de limpiar, necesitamos entender la estructura del dataset.


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Frecuencia de productos por categoría
df["category"].value_counts()


In [ ]:
# Frecuencia de productos por país
df["country"].value_counts()


## Ejercicio 3 · Tratamiento de valores faltantes

Los nulos no se corrigen siempre de la misma forma. Primero debemos detectarlos y evaluar su impacto.

Estrategias comunes:

- eliminar filas si el dato es crítico y son pocos casos;
- imputar con media o mediana si la columna es numérica;
- imputar con una categoría como `No informado` si la columna es textual;
- dejar el nulo si tiene significado analítico.


In [ ]:
# Conteo de valores faltantes por columna
df.isna().sum()


In [ ]:
# Filas que tienen al menos un valor faltante
filas_con_nulos = df[df.isna().any(axis=1)]
filas_con_nulos.head(10)


### Crear una copia de trabajo

No modificaremos el DataFrame original directamente. Crearemos una copia llamada `df_clean`.


In [ ]:
df_clean = df.copy()

# Imputar price_usd con la mediana porque puede haber outliers
mediana_precio = df_clean["price_usd"].median()
df_clean["price_usd"] = df_clean["price_usd"].fillna(mediana_precio)

# Imputar export_volume_liters con 0 si asumimos que no se registró exportación
# En un caso real, esta decisión debe validarse con negocio.
df_clean["export_volume_liters"] = df_clean["export_volume_liters"].fillna(0)

# Imputar rating con la media
media_rating = df_clean["rating"].mean()
df_clean["rating"] = df_clean["rating"].fillna(media_rating)

# Imputar distributor como No informado
df_clean["distributor"] = df_clean["distributor"].fillna("No informado")

# Imputar abv_percent con la mediana por categoría
mediana_abv_categoria = df_clean.groupby("category")["abv_percent"].transform("median")
df_clean["abv_percent"] = df_clean["abv_percent"].fillna(mediana_abv_categoria)

df_clean.isna().sum()


### Mini-análisis

Discute en parejas:

- ¿Qué imputaciones fueron razonables?
- ¿Cuál imputación te parece más discutible?
- ¿Qué pregunta harías al área de negocio antes de limpiar definitivamente?


## Ejercicio 4 · Detección de outliers con IQR

El método IQR usa cuartiles para identificar valores extremos.

```text
IQR = Q3 - Q1
Límite inferior = Q1 - 1.5 * IQR
Límite superior = Q3 + 1.5 * IQR
```

Aplicaremos el método a `price_usd`.


In [ ]:
q1 = df_clean["price_usd"].quantile(0.25)
q3 = df_clean["price_usd"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)


In [ ]:
outliers_precio = df_clean[
    (df_clean["price_usd"] < limite_inferior) |
    (df_clean["price_usd"] > limite_superior)
]

outliers_precio[["product_id", "country", "category", "brand", "price_usd"]]


In [ ]:
# Crear una versión sin outliers de precio para ciertos análisis comparativos
df_sin_outliers = df_clean[
    (df_clean["price_usd"] >= limite_inferior) &
    (df_clean["price_usd"] <= limite_superior)
]

print("Filas originales:", len(df_clean))
print("Filas sin outliers:", len(df_sin_outliers))


## Ejercicio 5 · Resúmenes con `groupby()`

`groupby()` permite resumir información por categorías.

Estructura mental:

```text
DataFrame → agrupar por una columna → calcular métricas → interpretar
```


In [ ]:
# Productos por país
df_clean.groupby("country")["product_id"].count().sort_values(ascending=False)


In [ ]:
# Precio promedio y rating promedio por categoría
resumen_categoria = (
    df_clean
    .groupby("category")
    .agg(
        productos=("product_id", "count"),
        precio_promedio=("price_usd", "mean"),
        rating_promedio=("rating", "mean"),
        exportacion_total_litros=("export_volume_liters", "sum")
    )
    .reset_index()
    .sort_values("exportacion_total_litros", ascending=False)
)

resumen_categoria


In [ ]:
# Resumen por país y canal de venta
resumen_pais_canal = (
    df_clean
    .groupby(["country", "sales_channel"])
    .agg(
        productos=("product_id", "count"),
        precio_promedio=("price_usd", "mean"),
        exportacion_total_litros=("export_volume_liters", "sum")
    )
    .reset_index()
)

resumen_pais_canal.head(10)


### Preguntas de análisis

1. ¿Qué categoría tiene mayor volumen total exportado?
2. ¿Qué país concentra más productos?
3. ¿Qué canal parece tener productos de mayor precio promedio?
4. ¿Cambiarían tus conclusiones si excluyes outliers de precio?


## Ejercicio 6 · Visualización básica con pandas

Usaremos `.plot()` para crear visualizaciones rápidas. No buscamos diseño final, sino apoyar la interpretación.


In [ ]:
# Gráfico de barras: exportación total por categoría
resumen_categoria.plot(
    x="category",
    y="exportacion_total_litros",
    kind="bar",
    legend=False,
    title="Exportación total por categoría"
)

plt.xlabel("Categoría")
plt.ylabel("Litros exportados")
plt.show()


In [ ]:
# Histograma de precios sin excluir outliers
df_clean["price_usd"].plot(
    kind="hist",
    bins=15,
    title="Distribución de precios"
)

plt.xlabel("Precio USD")
plt.show()


In [ ]:
# Gráfico de dispersión: precio vs porcentaje de alcohol
df_clean.plot(
    x="abv_percent",
    y="price_usd",
    kind="scatter",
    title="Relación entre ABV y precio"
)

plt.xlabel("ABV %")
plt.ylabel("Precio USD")
plt.show()


## Ejercicio final · Mini-reporte en Markdown

Crea una celda de texto y responde:

1. ¿Qué problema de calidad de datos encontraste?
2. ¿Qué decisión de limpieza tomaste?
3. ¿Qué hallazgo obtuviste con `groupby()`?
4. ¿Qué gráfica usarías para comunicar el resultado?
5. ¿Qué recomendación darías al equipo de negocio?

Formato sugerido:

```text
Hallazgo:
Evidencia:
Recomendación:
Limitación:
```


## Preguntas de validación de conocimiento

1. ¿Por qué es recomendable crear `df_clean = df.copy()` antes de modificar datos?
2. ¿Qué hace `.isna().sum()`?
3. ¿Cuándo usarías mediana en lugar de media para imputar valores?
4. ¿Qué identifica el método IQR?
5. ¿Qué problema resuelve `groupby()`?
6. ¿Por qué una gráfica no reemplaza la interpretación del analista?

### Respuestas esperadas

1. Para preservar el dataset original y poder comparar cambios.
2. Cuenta valores faltantes por columna.
3. Cuando hay valores extremos o distribución sesgada.
4. Posibles valores atípicos según cuartiles.
5. Permite resumir métricas por grupos o categorías.
6. Porque la gráfica muestra patrones, pero el analista debe explicar contexto, causa probable, impacto y limitaciones.


## Takeaways

1. Un análisis en pandas debe iniciar con exploración, no con limpieza inmediata.
2. Los nulos requieren decisiones justificadas según contexto de negocio.
3. La mediana es robusta frente a outliers.
4. IQR es una técnica sencilla para detectar valores extremos.
5. `groupby()` convierte datos detallados en métricas útiles.
6. `.plot()` permite validar visualmente patrones antes de construir un dashboard final.
7. Un notebook bien documentado comunica proceso, no solo código.


## Canales de ayuda

Recuerda los canales de ayuda disponibles:

- `DATA-CO-LEARNING`: revisa los horarios de atención en la guía del estudiante. Hay espacios de apoyo para resolver dudas puntuales.
- `DA_CONSULTA`: publica preguntas sobre el contenido de la plataforma o el proyecto. Incluye contexto, captura o fragmento de código y etiqueta `@Dataconsulta`.
- `SPRINT FOCUS`: para los Sprints 1 al 5 hay sesiones extra donde se profundizan temas del sprint. Revisa la guía del estudiante para conocer horarios.
- `SESIONES 1:1`: agenda sesiones individuales con antelación para resolver bloqueos específicos.

Cuando pidas ayuda sobre Python, incluye siempre: qué querías lograr, el código que ejecutaste, el error completo y una captura o copia del resultado.


## Siguientes pasos

Para practicar después de clase:

1. Cambia el análisis de outliers de `price_usd` a `export_volume_liters`.
2. Crea un resumen por `sales_channel`.
3. Filtra solo productos premium y compara su precio promedio por país.
4. Escribe un mini-reporte ejecutivo de 5 líneas con tu hallazgo principal.
